In [2]:
pip install pyspark

     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     ---------------------------------------- 0.3/455.4 MB ? eta -:--:--
     --------------------------------------- 3.4/455.4 MB 14.5 MB/s eta 0:00:32
      -------------------------------------- 8.1/455.4 MB 18.2 MB/s eta 0:00:25
     - ------------------------------------ 13.1/455.4 MB 20.2 MB/s eta 0:00:22
     - ------------------------------------ 17.6/455.4 MB 20.6 MB/s eta 0:00:22
     - ------------------------------------ 22.5/455.4 MB 21.4 MB/s eta 0:00:21
     -- ----------------------------------- 27.8/455.4 MB 22.2 MB/s eta 0:00:20
     -- ----------------------------------- 33.6/455.4 MB 23.1 MB/s eta 0:00:19
     --- ---------------------------------- 38.5/455.4 MB 23.3 MB/s eta 0:00:18
     --- ---------------------------------- 43.5/455.4 MB 23.4 MB/s eta 0:00:18
     ---- --------------------------------- 48.5/455.4 MB 23.5 MB/s eta 0:00:18
     ---- --------------------------------- 54.3/455.4


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pyspark
print(pyspark.__version__)

4.1.1


#### Step 11 - PySpark Big Data Analysis
##### Input: enhanced_collision.csv
##### Process: Load and analyse data using PySpark distributed framework
##### Output: Schema, class distribution, and Logistic Regression using MLlib

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("UKRoadAccidentSeverity") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark session started")

Spark version: 4.1.1
Spark session started


In [7]:
df_spark = spark.read.csv(
    '../Data/enhanced_collision.csv',
    header=True,
    inferSchema=True
)

print("Shape:", (df_spark.count(), len(df_spark.columns)))
df_spark.printSchema()

Shape: (503373, 30)
root
 |-- collision_year: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- collision_severity: integer (nullable = true)
 |-- number_of_vehicles: integer (nullable = true)
 |-- number_of_casualties: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- first_road_class: integer (nullable = true)
 |-- road_type: integer (nullable = true)
 |-- speed_limit: double (nullable = true)
 |-- junction_detail: double (nullable = true)
 |-- second_road_class: double (nullable = true)
 |-- pedestrian_crossing: double (nullable = true)
 |-- light_conditions: double (nullable = true)
 |-- weather_conditions: double (nullable = true)
 |-- road_surface_conditions: double (nullable = true)
 |-- special_conditions_at_site: double (nullable = true)
 |-- carriageway_hazards: double (nullable = true)
 |-- urban_or_rural_area: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- month: integ

In [8]:
print("Class Distribution:")
df_spark.groupBy('collision_severity') \
    .count() \
    .orderBy('collision_severity') \
    .show()

Class Distribution:
+------------------+------+
|collision_severity| count|
+------------------+------+
|                 1|  7486|
|                 2|109952|
|                 3|385935|
+------------------+------+



In [9]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# convert target to 0 based index because MLlib needs 0,1,2 not 1,2,3
from pyspark.sql.functions import col
df_spark = df_spark.withColumn('label', col('collision_severity') - 1)

# same feature columns we used before
feature_cols = [
    'longitude', 'latitude', 'number_of_vehicles', 'number_of_casualties',
    'day_of_week', 'first_road_class', 'road_type', 'speed_limit',
    'junction_detail', 'second_road_class', 'pedestrian_crossing',
    'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area',
    'hour', 'month', 'is_rush_hour', 'is_weekend', 'is_dark',
    'is_bad_weather', 'is_highspeed', 'is_urban', 'is_junction', 'is_hazards'
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
df_assembled = assembler.transform(df_spark)
df_final = df_assembled.select('features', 'label')

print("Features assembled")
df_final.show(5)

Features assembled
+--------------------+-----+
|            features|label|
+--------------------+-----+
|[-1.270905,54.689...|    2|
|[-1.218333,54.690...|    2|
|[-1.232884,54.570...|    2|
|(27,[0,1,2,3,4,5,...|    2|
|[-1.290292,54.580...|    2|
+--------------------+-----+
only showing top 5 rows


In [10]:
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

lr_spark = LogisticRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    family='multinomial'
)

model_spark = lr_spark.fit(train_df)
print("PySpark Logistic Regression trained")

PySpark Logistic Regression trained


In [11]:
predictions = model_spark.transform(test_df)

evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)

accuracy = evaluator.evaluate(predictions)
print(f"PySpark Model Accuracy: {accuracy:.4f}")

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
)

f1 = evaluator_f1.evaluate(predictions)
print(f"PySpark Model F1 Score: {f1:.4f}")

PySpark Model Accuracy: 0.7666
PySpark Model F1 Score: 0.6681
